# Demo: Event Summarization using SmolVLM

This notebook provides a demonstration for the second task of the assignment. Given the strong performance of vision-language models across a variety of tasks, we use an efficient, lightweight model: [SmolVLM 2](https://huggingface.co/blog/smolvlm2)
The goal of this notebook is to guide you on how to: design prompts for event detection, process model outputs, and evaluate the quality of detected events.
You are expected to build upon this baseline and critically evaluate your chosen approaches.

In [ ]:
!pip install num2words

In [ ]:
!pip install av


SmolVLM2 is a lightweight vision-language model designed for efficient video understanding with relatively few parameters. The material presented here is largely based on the original documentation and is intended to provide a starting point for developing your own approach.

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
import cv2
from PIL import Image
import random
model_path = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
processor = AutoProcessor.from_pretrained(model_path)
model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
).to("cuda")

Since SmolVLM2 is a vision-language model (VLM), you will need to design an appropriate prompt for the task. The prompt can be flexible and should be adapted based on the type of output you want (e.g., event lists, descriptions, or timestamps).

Vision-language models process visual inputs (such as frames or videos) together with text by encoding them into a shared representation space. The exact input format may vary across models.

In our case, the HuggingFace implementation handles most of the input preprocessing (e.g., frame sampling and formatting), allowing you to focus primarily on prompt design and output parsing.

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "video", "path": "video_1.mp4"},
            {"type": "text", "text": "Describe the most relevant events in the video, list each event sequentially, using a numbered format. Describe it in terms of the actions different persons take"}
        ]
    },
]

inputs = processor.apply_chat_template(
    messages,
    num_frames = 15,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device, dtype=torch.bfloat16) # Considering the size of the model, and the number of frames we are sampling are few. This is ultimately a choice that you must make
generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=128)
generated_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)

print(generated_texts[0])

The detected events are often highly descriptive, but they do not always capture the most important or salient moments in the video.

To address this, you may explore alternative strategies. One approach is to use a language model (LLM) to process frame-level descriptions generated from the video.

For example, you can experiment with randomly sampling frames from the video, generating descriptions for each frame using a vision-language model (VLM), and then aggregating these descriptions into a structured set of events using an LLM.


In [ ]:
def sample_frames(video_path, num_frames):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        raise ValueError("Video has no frames.")

    frame_indices = sorted(random.sample(range(total_frames), min(num_frames, total_frames)))

    frames = []
    current_idx = 0
    target_idx_set = set(frame_indices)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if current_idx in target_idx_set:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(frame))

        current_idx += 1

    cap.release()
    return frames

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": frames[0]},
            {"type": "image", "url": frames[1]},
            {"type": "image", "url": frames[2]},
            {"type": "image", "url": frames[3]},
            {"type": "text", "text": "Describe the most relevant events in the video, list each event sequentially, using a numbered format. Describe it in terms of the actions different persons take"}
        ]
    },
]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)

You may improve event detection by splitting the video into smaller temporal segments and prompting the VLM on each segment separately. The resulting partial event descriptions can then be combined into a single, coherent list using an LLM.

Crucially, you must define an evaluation scheme for event coverage, for example:
-How many annotated events are correctly detected?
-Which events are missed or incorrectly predicted?
-How does performance change with different prompting or segmentation strategies?

In [ ]:
generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=256)

generated_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)

print(generated_texts[0])

The generated outputs may be quite descriptive. While this can introduce noise, it also provides an opportunity to infer potential events through reasoning over these descriptions.

Therefore, one possible strategy is to first generate scene-level descriptions using a vision-language model (VLM), and then further process these descriptions using a language model (LLM) to extract structured events.

The choice of strategy is left to you. You are expected to design and implement an approach that produces meaningful event representations from the video.

---

# Part 2 — Two-Stage VLM + LLM Event Detection Pipeline

**Strategy:** Sample frames uniformly from the video → generate a per-frame description with SmolVLM2 (Stage 1) → feed all descriptions to the same model in text-only mode to extract a structured event list (Stage 2).

This section processes all four videos (`video_21` – `video_24`) end-to-end and saves results to `outputs/`.

## Section A — Setup

Install and import all required libraries.

In [ ]:
!pip install transformers accelerate --quiet
!pip install torch torchvision torchaudio --quiet
!pip install opencv-python pillow pandas num2words av --quiet

In [ ]:
import re
import json
import torch
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from transformers import AutoProcessor, AutoModelForImageTextToText

print("All libraries imported successfully!")

## Section B — Configuration

Set the list of videos to process and all pipeline parameters here.

In [ ]:
# ── Videos to process ────────────────────────────────────────────────────────
VIDEOS = ["video_21.mp4", "video_22.mp4", "video_23.mp4", "video_24.mp4"]

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"

# ── Token budgets ─────────────────────────────────────────────────────────────
MAX_TOKENS_FRAME = 80    # per-frame description (Stage 1)
MAX_TOKENS_EVENT = 512   # event list (Stage 2)

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def compute_num_frames(duration_sec):
    """Adaptive frame count scaled to video length."""
    minutes = duration_sec / 60
    if minutes <= 1:   return 8
    elif minutes <= 3: return 12
    elif minutes <= 10: return 16
    else:              return 24


def compute_target_events(duration_sec):
    """Target number of distinct salient events scaled to video length."""
    minutes = duration_sec / 60
    if minutes <= 2:   return 5
    elif minutes <= 5: return 8
    elif minutes <= 10: return 12
    else:              return 15


print("Configuration:")
print(f"  Model  : {MODEL_NAME}")
print(f"  Videos : {VIDEOS}")
print(f"  Output : {OUTPUT_DIR}")
print()
print("Adaptive scaling preview:")
for d, label in [(60, "1 min"), (180, "3 min"), (420, "7 min"), (720, "12 min")]:
    print(f"  {label:>6}  →  {compute_num_frames(d):>2} frames  |  {compute_target_events(d)} target events")

## Section C — Video Loading & Frame Sampling

We inspect each video to get its duration and FPS, then sample frames **uniformly** across the full video using direct frame seeking — this is fast even for long videos and always covers the complete duration.

In [ ]:
def get_video_info(video_path):
    """Return (fps, total_frames, duration_seconds)."""
    cap   = cv2.VideoCapture(str(video_path))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    dur   = total / fps if fps > 0 else 0.0
    cap.release()
    return fps, total, dur


def secs_to_mmss(s):
    m, sec = divmod(int(s), 60)
    return f"{m:02d}:{sec:02d}"


def sample_frames_uniform(video_path, n_frames):
    """
    Sample exactly n_frames spread uniformly across the full video via direct
    frame seeking.  Returns (frames: list[PIL.Image], timestamps_sec: list[float]).

    For a 10-min video with n_frames=16 the interval is ~37 s:
        Frame  1  →  0:00
        Frame  2  →  0:37
        ...
        Frame 16  →  9:23
    """
    cap   = cv2.VideoCapture(str(video_path))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0 or fps == 0:
        cap.release()
        return [], []

    indices = [int(total * i / n_frames) for i in range(n_frames)]
    frames, timestamps = [], []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
            timestamps.append(round(idx / fps, 2))
    cap.release()

    interval = (total / fps) / n_frames
    print(f"  Sampled {len(frames)} frames | ~{interval:.0f}s interval | "
          f"{secs_to_mmss(timestamps[0])} → {secs_to_mmss(timestamps[-1])}")
    return frames, timestamps


# ── Preview all videos ────────────────────────────────────────────────────────
print("Video summary:")
for v in VIDEOS:
    if Path(v).exists():
        fps, tot, dur = get_video_info(v)
        print(f"  {v}: {dur:.0f}s ({dur/60:.1f} min) | "
              f"{compute_num_frames(dur)} frames | "
              f"{compute_target_events(dur)} target events")
    else:
        print(f"  {v}: NOT FOUND")

## Section D — Load the Vision-Language Model

We load **SmolVLM2-2.2B-Instruct** from HuggingFace.

- GPU available → loaded in **bfloat16** (half precision, faster)
- CPU only → loaded in **float32** (slower but works)

> First download is ~4 GB. Subsequent runs use the local HuggingFace cache.

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if DEVICE == "cuda" else torch.float32

print(f"Loading '{MODEL_NAME}' on {DEVICE} ({DTYPE}) ...")

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
).to(DEVICE)

print("Model loaded successfully!")

## Section E — Stage 1: VLM Per-Frame Descriptions

Each uniformly sampled frame is passed to SmolVLM2 **individually** with a brief prompt asking for a 1–2 sentence description.

The result is a list of `(MM:SS, description)` pairs — one entry per sampled frame — that capture what the model sees at each point in time.

This per-frame granularity is the key advantage over the direct-video approach: each frame gets its own focused description, which Stage 2 can reason over precisely.

In [ ]:
FRAME_DESC_PROMPT = (
    "Describe only what is happening in this single video frame. "
    "Focus on people, their actions, and the scene context. "
    "Be concise: at most 2 sentences, under 40 words."
)


def describe_frames(frames, timestamps):
    """
    Run SmolVLM2 on each frame individually → per-frame descriptions.
    Returns list of (timestamp_mmss: str, description: str).
    """
    results = []
    for frame, ts_sec in zip(frames, timestamps):
        ts_label = secs_to_mmss(ts_sec)
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "url": frame},
                {"type": "text",  "text": FRAME_DESC_PROMPT},
            ],
        }]
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(DEVICE, dtype=DTYPE)

        with torch.no_grad():
            ids = model.generate(**inputs, do_sample=False, max_new_tokens=MAX_TOKENS_FRAME)

        raw  = processor.batch_decode(ids, skip_special_tokens=True)[0]
        desc = raw.split("Assistant:")[-1].strip() if "Assistant:" in raw else raw.strip()
        results.append((ts_label, desc))
        print(f"  [{ts_label}] {desc[:100]}")

    return results


print("describe_frames() defined — Stage 1 ready.")

## Section F — Stage 2: LLM Event Extraction from Descriptions

All per-frame descriptions are concatenated into a single text block (with timestamps) and passed to SmolVLM2 in **text-only mode** — no images are sent at this stage.

The model is instructed to:
- Group consecutive similar frames into **one** event
- Eliminate repeated action types
- Return exactly *N* salient events with `MM:SS – MM:SS` timestamp ranges

*N* is determined by video duration (same scale as Section B).

In [ ]:
def build_llm_prompt(frame_descriptions, duration_sec):
    """Construct the text-only LLM aggregation prompt."""
    n_events   = compute_target_events(duration_sec)
    dur_str    = f"{duration_sec / 60:.1f} minutes"
    desc_block = "\n".join(f"[{ts}]: {desc}" for ts, desc in frame_descriptions)
    ts_list    = ", ".join(ts for ts, _ in frame_descriptions)

    return (
        f"You are given frame-by-frame descriptions of a {dur_str} video:\n\n"
        f"{desc_block}\n\n"
        f"Your task: extract exactly {n_events} salient events from these descriptions.\n\n"
        f"RULES:\n"
        f"- A salient event is a meaningful change, action, or transition.\n"
        f"- Group consecutive frames with similar content into ONE event.\n"
        f"- NEVER repeat the same action type (e.g. 'person walks' appears at most once).\n"
        f"- Every event must be fundamentally different from all others.\n"
        f"- Use MM:SS - MM:SS timestamp ranges using ONLY values from: {ts_list}\n\n"
        f"OUTPUT FORMAT — strictly one event per line:\n"
        f"Event 1: <description>, MM:SS - MM:SS\n"
        f"Event 2: <description>, MM:SS - MM:SS\n"
        f"...\n"
        f"Event {n_events}: <description>, MM:SS - MM:SS\n\n"
        f"Generate EXACTLY {n_events} events, no more, no less."
    )


def extract_events_with_llm(frame_descriptions, duration_sec):
    """
    Run SmolVLM2 in text-only mode to extract a structured event list
    from the aggregated frame descriptions.
    Returns the raw output string.
    """
    prompt   = build_llm_prompt(frame_descriptions, duration_sec)
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inputs   = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(DEVICE, dtype=DTYPE)

    with torch.no_grad():
        ids = model.generate(**inputs, do_sample=False, max_new_tokens=MAX_TOKENS_EVENT)

    raw = processor.batch_decode(ids, skip_special_tokens=True)[0]
    return raw.split("Assistant:")[-1].strip() if "Assistant:" in raw else raw.strip()


print("build_llm_prompt() and extract_events_with_llm() defined — Stage 2 ready.")

## Section G — Output Parser

Converts the raw LLM text into a structured **pandas DataFrame**.

Columns: `event_number`, `video`, `description`, `start_time`, `end_time`, `start_seconds`, `end_seconds`

Primary pattern: `Event N: description, MM:SS - MM:SS`
Fallback pattern: `N. description, MM:SS - MM:SS`

In [ ]:
def parse_events(raw_text):
    """
    Extract (description, start_mmss, end_mmss) tuples from LLM output.
    Tries two line formats; returns empty list if neither matches.
    """
    ts_re    = r"(\d{1,2}:\d{2})\s*[-–]\s*(\d{1,2}:\d{2})"
    pat_ev   = re.compile(r"Event\s+\d+\s*:\s*(.+?),\s*" + ts_re, re.IGNORECASE)
    pat_num  = re.compile(r"^\d+\.\s+(.+?),\s*" + ts_re)

    rows = []
    for line in raw_text.splitlines():
        m = pat_ev.search(line.strip())
        if m:
            rows.append((m.group(1).strip(), m.group(2), m.group(3)))

    if not rows:
        for line in raw_text.splitlines():
            m = pat_num.match(line.strip())
            if m:
                rows.append((m.group(1).strip(), m.group(2), m.group(3)))
    return rows


def _ts_to_sec(ts):
    try:
        p = [int(x) for x in ts.split(":")]
        return p[0] * 60 + p[1] if len(p) == 2 else p[0] * 3600 + p[1] * 60 + p[2]
    except Exception:
        return None


def build_dataframe(rows, video_name):
    """Convert parsed rows to a tidy DataFrame."""
    return pd.DataFrame([
        {
            "event_number":  i,
            "video":         video_name,
            "description":   desc,
            "start_time":    start,
            "end_time":      end,
            "start_seconds": _ts_to_sec(start),
            "end_seconds":   _ts_to_sec(end),
        }
        for i, (desc, start, end) in enumerate(rows, 1)
    ])


print("parse_events() and build_dataframe() defined.")

## Section H — Full Pipeline

`process_video()` ties both stages together for a single video:

| Step | What happens |
|------|-------------|
| 1 | Get video info (fps, duration) |
| 2 | Sample *N* frames uniformly across full video |
| 3 | **Stage 1** — VLM describes each frame individually |
| 4 | **Stage 2** — LLM aggregates descriptions into structured events |
| 5 | Parse raw output into a DataFrame |

In [ ]:
def process_video(video_path):
    """
    Full two-stage VLM+LLM pipeline for one video file.
    Returns (events_df, frame_descriptions, raw_llm_output).
    """
    video_path = str(video_path)
    _, _, duration = get_video_info(video_path)
    n_frames   = compute_num_frames(duration)
    n_events   = compute_target_events(duration)

    print(f"\n{'='*65}")
    print(f"VIDEO : {video_path}")
    print(f"  Duration : {duration:.1f}s  ({duration/60:.1f} min)")
    print(f"  Frames   : {n_frames} sampled  |  Target events: {n_events}")
    print(f"{'='*65}")

    # ── Stage 1: per-frame VLM descriptions ──────────────────────────────────
    print("\nStage 1 — VLM per-frame descriptions:")
    frames, timestamps = sample_frames_uniform(video_path, n_frames)
    frame_descs = describe_frames(frames, timestamps)

    # ── Stage 2: LLM event extraction (text-only) ─────────────────────────────
    print("\nStage 2 — LLM event extraction (text-only):")
    raw_output = extract_events_with_llm(frame_descs, duration)
    print(raw_output)

    # ── Parse into DataFrame ──────────────────────────────────────────────────
    rows = parse_events(raw_output)
    if rows:
        df = build_dataframe(rows, Path(video_path).name)
    else:
        print("WARNING: parser found no events — check raw output above.")
        df = pd.DataFrame(columns=[
            "event_number", "video", "description",
            "start_time", "end_time", "start_seconds", "end_seconds"
        ])

    print(f"\n→ {len(df)} events extracted from {Path(video_path).name}")
    return df, frame_descs, raw_output


print("process_video() defined — full pipeline ready.")

## Section I — Run the Pipeline on All Videos

> **Note:** Each video takes ~4–8 min on a T4 GPU (Stage 1 = one VLM call per frame; Stage 2 = one text-only call). On CPU expect ~30–60 min per video.

In [ ]:
all_results = {}  # video → {"df": ..., "frame_descriptions": ..., "raw_llm_output": ...}

for video in VIDEOS:
    if not Path(video).exists():
        print(f"SKIP: {video} not found.")
        continue
    df, descs, raw = process_video(video)
    all_results[video] = {
        "df":                 df,
        "frame_descriptions": descs,
        "raw_llm_output":     raw,
    }

print("\n" + "="*65)
print("All videos processed:")
for v, r in all_results.items():
    print(f"  {v}: {len(r['df'])} events extracted")

## Section J — Save Outputs

| File | Contents |
|------|----------|
| `<stem>_vlm_llm_events.txt` | Frame descriptions + raw LLM output |
| `<stem>_vlm_llm_events.csv` | Parsed event table (per video) |
| `all_videos_vlm_llm_events.csv` | Combined table for all videos |
| `vlm_llm_pipeline_results.json` | Full metadata + events (JSON) |

In [ ]:
for video, res in all_results.items():
    stem  = Path(video).stem
    df    = res["df"]
    descs = res["frame_descriptions"]
    raw   = res["raw_llm_output"]

    # Plain-text summary
    txt_path = OUTPUT_DIR / f"{stem}_vlm_llm_events.txt"
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(f"Video: {video}\n{'='*60}\n")
        f.write("STAGE 1 — Frame Descriptions:\n")
        for ts, desc in descs:
            f.write(f"  [{ts}] {desc}\n")
        f.write(f"\nSTAGE 2 — Raw LLM Output:\n{raw}\n")

    # Per-video CSV
    csv_path = OUTPUT_DIR / f"{stem}_vlm_llm_events.csv"
    df.to_csv(csv_path, index=False)
    print(f"Saved: {txt_path}  |  {csv_path}")

# Combined CSV
all_dfs = [r["df"] for r in all_results.values() if len(r["df"]) > 0]
if all_dfs:
    combined_csv = OUTPUT_DIR / "all_videos_vlm_llm_events.csv"
    pd.concat(all_dfs, ignore_index=True).to_csv(combined_csv, index=False)
    print(f"Combined CSV: {combined_csv}")

# Full JSON
json_payload = {}
for video, res in all_results.items():
    _, _, dur = get_video_info(video)
    json_payload[video] = {
        "duration_seconds":   round(dur, 2),
        "frames_sampled":     len(res["frame_descriptions"]),
        "target_events":      compute_target_events(dur),
        "events_found":       len(res["df"]),
        "frame_descriptions": [{"timestamp": ts, "description": d}
                                for ts, d in res["frame_descriptions"]],
        "raw_llm_output": res["raw_llm_output"],
        "events": res["df"].to_dict(orient="records"),
    }

json_path = OUTPUT_DIR / "vlm_llm_pipeline_results.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_payload, f, indent=2, ensure_ascii=False)
print(f"JSON : {json_path}")
print("\nAll outputs saved to outputs/")

## Section K — Display Results

In [ ]:
for video, res in all_results.items():
    df = res["df"]
    print(f"\n{'='*65}")
    print(f"VIDEO: {video}  ({len(df)} events)")
    print(f"{'='*65}")
    if len(df) > 0:
        display(df[["event_number", "start_time", "end_time", "description"]])
    else:
        print("  No events could be parsed from this video.")